# Preprocess SSP Population and GDP Inputs

Resamples SSP population and GDP files from their native high resolution (~0.008Â°, ~1 km) to 0.10Â° and applies log1p z-score normalization, matching the preprocessing used for the training data (`fix_data_pop_gdp.ipynb`).

Run this once before `projections.ipynb`. Output goes to `READY_data/inputs_normalized/ssp/`.

## 1. Setup â€” load reference grid

Loads the CISI label raster to extract the target 0.10Â° grid (extent, CRS, affine transform). All SSP inputs will be reprojected onto this exact grid so every pixel aligns with the training labels.

In [5]:
import os
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

OUTPUT_DIR   = r'READY_data\inputs_normalized\ssp'
REFERENCE    = r'READY_data\labels\2024_CISI_010deg_nearest.tif'  # defines the 0.10Â° grid
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load reference grid
with rasterio.open(REFERENCE) as ref:
    ref_crs       = ref.crs
    ref_transform = ref.transform
    ref_height    = ref.height
    ref_width     = ref.width
    ref_profile   = ref.profile.copy()

print(f'Reference grid: {ref_height}x{ref_width}, res={ref_transform[0]:.3f}Â°')

Reference grid: 485x570, res=0.100Â°


## 2. Resample and normalize function

Defines the core processing function applied to every input file:
1. **Reproject** from native resolution to 0.10Â° using sum aggregation (preserves total population/GDP per cell)
2. **Clean nodata** â€” removes sentinel values like `-3.4e38` that are not declared in file metadata
3. **log1p z-score normalize** â€” compresses the skewed distribution and centres it, matching what was done to the training inputs in `fix_data_pop_gdp.ipynb`

In [6]:
def resample_and_normalize(input_path, output_path):
    """Resample to 0.10Â° grid then apply log1p z-score normalization."""
    print(f'Processing: {os.path.basename(input_path)}')

    # Step 1: Resample to reference grid (sum aggregation for population)
    data = np.full((ref_height, ref_width), np.nan, dtype=np.float32)
    with rasterio.open(input_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=data,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=Resampling.sum,
        )

    # Step 2: Clean nodata flags
    for flag in [-3.4028235e+38, -3.402823e+38, -9999, -32768]:
        data[np.isclose(data, flag, rtol=1e-5)] = np.nan
    data[data < -1e10] = np.nan
    data[np.isinf(data)] = np.nan

    valid = ~np.isnan(data)
    print(f'  Valid pixels after resample: {valid.sum():,}')

    # Step 3: log1p z-score normalization
    log_data = np.log1p(np.abs(data[valid]))
    mean_val = log_data.mean()
    std_val  = log_data.std()

    normalized = np.full_like(data, np.nan)
    normalized[valid] = (np.log1p(np.abs(data[valid])) - mean_val) / (std_val + 1e-8)

    print(f'  Normalized range: [{np.nanmin(normalized):.3f}, {np.nanmax(normalized):.3f}]  mean={np.nanmean(normalized):.4f}')

    # Step 4: Save
    profile = ref_profile.copy()
    profile.update(dtype='float32', count=1, nodata=np.nan)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(normalized[np.newaxis, :, :])
    print(f'  Saved -> {output_path}')

## 3. Process population files (15 files)

Runs the resample+normalize function on all 15 SSP population rasters (SSP1â€“5 Ã— 2030/2050/2100). Source files are at ~0.008Â° (~1 km). Output: `POP_SSP{n}_{year}_normalized.tif`. Skips files that already exist.

In [7]:
# All 15 SSP population files
POP_FILES = {
    'SSP1': {
        2030: r'READY_data\SSP1\SSP1_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP1\SSP1_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP1\SSP1_2100_EU_UK_POP_01.tif',
    },
    'SSP2': {
        2030: r'READY_data\SSP2\SSP2_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP2\SSP2_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP2\SSP2_2100_EU_UK_POP_01.tif',
    },
    'SSP3': {
        2030: r'READY_data\SSP3\SSP3_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP3\SSP3_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP3\SSP3_2100_EU_UK_POP_01.tif',
    },
    'SSP4': {
        2030: r'READY_data\SSP4\SSP4_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP4\SSP4_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP4\SSP4_2100_EU_UK_POP_01.tif',
    },
    'SSP5': {
        2030: r'READY_data\SSP5\SSP5_2030_EU_UK_POP_01.tif',
        2050: r'READY_data\SSP5\SSP5_2050_EU_UK_POP_01.tif',
        2100: r'READY_data\SSP5\SSP5_2100_EU_UK_POP_01.tif',
    },
}

for ssp, years in POP_FILES.items():
    for year, in_path in years.items():
        out_path = os.path.join(OUTPUT_DIR, f'POP_{ssp}_{year}_normalized.tif')
        if os.path.exists(out_path):
            print(f'Skipping {ssp} {year} pop (already exists)')
            continue
        resample_and_normalize(in_path, out_path)

print('\nAll population files processed.')

Processing: SSP1_2030_EU_UK_POP_01.tif
  Valid pixels after resample: 276,450
  Normalized range: [-4.116, 1.653]  mean=-0.0000
  Saved -> READY_data\inputs_normalized\ssp\POP_SSP1_2030_normalized.tif
Processing: SSP1_2050_EU_UK_POP_01.tif
  Valid pixels after resample: 276,450
  Normalized range: [-3.997, 1.641]  mean=-0.0000
  Saved -> READY_data\inputs_normalized\ssp\POP_SSP1_2050_normalized.tif
Processing: SSP1_2100_EU_UK_POP_01.tif
  Valid pixels after resample: 276,450
  Normalized range: [-3.708, 1.570]  mean=0.0000
  Saved -> READY_data\inputs_normalized\ssp\POP_SSP1_2100_normalized.tif
Processing: SSP2_2030_EU_UK_POP_01.tif
  Valid pixels after resample: 276,450
  Normalized range: [-4.110, 1.651]  mean=0.0000
  Saved -> READY_data\inputs_normalized\ssp\POP_SSP2_2030_normalized.tif
Processing: SSP2_2050_EU_UK_POP_01.tif
  Valid pixels after resample: 276,450
  Normalized range: [-3.988, 1.638]  mean=-0.0000
  Saved -> READY_data\inputs_normalized\ssp\POP_SSP2_2050_normalized.t

## 4. Resolution check â€” verify all outputs

Checks the resolution of raw inputs vs. normalized outputs. Confirms all 30 output files exist and are at 0.10Â°. Run this after processing to verify everything is correct before running `projections.ipynb`.

In [8]:
import os
import rasterio

_NORM_DIR = r'READY_data\inputs_normalized\ssp'
_SSPS  = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']
_YEARS = [2030, 2050, 2100]

# --- Raw input resolution check ---
print('=== RAW INPUT RESOLUTIONS ===')
raw_files = {
    'CISI label (target)':     r'READY_data\labels\2024_CISI_010deg_nearest.tif',
    'GDP training (2019)':     r'READY_data\inputs\2019_gdp_aligned_010.tif',
    'Pop training (2020)':     r'READY_data\inputs\2020_pop_aligned_010.tif',
    'Land cover (2020)':       r'READY_data\landuse_onehot\clipped_history_2020_onehot.tif',
    'GDP SSP raw (SSP1 2030)': r'READY_data\GDP SSP\GDP2030_ssp1.tif',
    'Pop SSP raw (SSP1 2030)': r'READY_data\SSP1\SSP1_2030_EU_UK_POP_01.tif',
}
print(f'  {"File":<35} {"Shape":>12} {"Res (Â°)":>10}  {"0.1Â° match?"}')
print('  ' + '-' * 75)
for label, path in raw_files.items():
    if not os.path.exists(path):
        print(f'  {label:<35} FILE NOT FOUND')
        continue
    with rasterio.open(path) as src:
        res_x = abs(src.transform[0])
        shape = f'{src.height}x{src.width}'
    ok = 'âœ“' if abs(res_x - 0.1) < 0.001 else f'âœ—  ({res_x:.5f}Â° â€” will be resampled)'
    print(f'  {label:<35} {shape:>12} {res_x:>10.5f}  {ok}')

# --- All 30 normalized output files ---
print('\n=== NORMALIZED OUTPUT FILES (all 30) ===')
print(f'  {"File":<40} {"Shape":>12} {"Res (Â°)":>10}  {"Status"}')
print('  ' + '-' * 80)

all_ok = True
for ssp in _SSPS:
    for year in _YEARS:
        for kind in ['POP', 'GDP']:
            fname = f'{kind}_{ssp}_{year}_normalized.tif'
            path  = os.path.join(_NORM_DIR, fname)
            if not os.path.exists(path):
                print(f'  {fname:<40} MISSING')
                all_ok = False
                continue
            with rasterio.open(path) as src:
                res_x = abs(src.transform[0])
                shape = f'{src.height}x{src.width}'
            ok = 'âœ“' if abs(res_x - 0.1) < 0.001 else f'âœ—  wrong res: {res_x:.5f}Â°'
            print(f'  {fname:<40} {shape:>12} {res_x:>10.5f}  {ok}')
            if abs(res_x - 0.1) >= 0.001:
                all_ok = False

print()
print('All 30 files present and at 0.1Â°.' if all_ok else 'WARNING: some files are missing or at wrong resolution.')

=== RAW INPUT RESOLUTIONS ===
  File                                       Shape    Res (Â°)  0.1Â° match?
  ---------------------------------------------------------------------------
  CISI label (target)                      485x570    0.10000  âœ“
  GDP training (2019)                      485x570    0.10000  âœ“
  Pop training (2020)                      485x570    0.10000  âœ“
  Land cover (2020)                        485x570    0.10000  âœ“
  GDP SSP raw (SSP1 2030)              18000x43200    0.00833  âœ—  (0.00833Â° â€” will be resampled)
  Pop SSP raw (SSP1 2030)                4554x8480    0.00833  âœ—  (0.00833Â° â€” will be resampled)

=== NORMALIZED OUTPUT FILES (all 30) ===
  File                                            Shape    Res (Â°)  Status
  --------------------------------------------------------------------------------
  POP_SSP1_2030_normalized.tif                  485x570    0.10000  âœ“
  GDP_SSP1_2030_normalized.tif             MISSING
  POP_SSP1_2050_nor

## 5. Process GDP files (15 files)

Same as step 3 but for GDP. Source files are in `READY_data/GDP SSP/` at ~0.008Â° resolution. Output: `GDP_SSP{n}_{year}_normalized.tif`. Skips files that already exist.

In [ ]:
GDP_FILES = {
    'SSP1': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp1.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp1.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp1.tif',
    },
    'SSP2': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp2.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp2.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp2.tif',
    },
    'SSP3': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp3.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp3.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp3.tif',
    },
    'SSP4': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp4.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp4.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp4.tif',
    },
    'SSP5': {
        2030: r'READY_data\GDP SSP\GDP2030_ssp5.tif',
        2050: r'READY_data\GDP SSP\GDP2050_ssp5.tif',
        2100: r'READY_data\GDP SSP\GDP2100_ssp5.tif',
    },
}

for ssp, years in GDP_FILES.items():
    for year, in_path in years.items():
        out_path = os.path.join(OUTPUT_DIR, f'GDP_{ssp}_{year}_normalized.tif')
        if os.path.exists(out_path):
            print(f'Skipping {ssp} {year} GDP (already exists)')
            continue
        resample_and_normalize(in_path, out_path)

print('\nAll GDP files processed.')

Processing: GDP2030_ssp1.tif


## 6. Verify GDP outputs

In [10]:
import os
import numpy as np
import rasterio

_NORM_DIR = r'READY_data\inputs_normalized\ssp'
_SSPS  = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']
_YEARS = [2030, 2050, 2100]

print(f'{"File":<35} {"Shape":>12} {"Valid px":>10} {"Min":>8} {"Max":>8} {"Mean":>8}  {"OK?"}')
print('-' * 100)

all_ok = True
for ssp in _SSPS:
    for year in _YEARS:
        fname = f'GDP_{ssp}_{year}_normalized.tif'
        path  = os.path.join(_NORM_DIR, fname)
        if not os.path.exists(path):
            print(f'{fname:<35} MISSING')
            all_ok = False
            continue
        with rasterio.open(path) as src:
            data  = src.read(1).astype(np.float32)
            res_x = abs(src.transform[0])
            shape = f'{src.height}x{src.width}'
        valid = ~np.isnan(data)
        res_ok    = abs(res_x - 0.1) < 0.001
        has_data  = valid.sum() > 0
        no_extremes = np.all(np.abs(data[valid]) < 20) if has_data else False
        ok = '✓' if (res_ok and has_data and no_extremes) else '✗'
        if not (res_ok and has_data and no_extremes):
            all_ok = False
        print(f'{fname:<35} {shape:>12} {valid.sum():>10,} {np.nanmin(data):>8.3f} {np.nanmax(data):>8.3f} {np.nanmean(data):>8.4f}  {ok}')

print()
print('All GDP files OK.' if all_ok else 'WARNING: one or more GDP files have issues.')

File                                       Shape   Valid px      Min      Max     Mean  OK?
----------------------------------------------------------------------------------------------------
GDP_SSP1_2030_normalized.tif             485x570    276,450   -0.663    2.845   0.0000  ✓
GDP_SSP1_2050_normalized.tif             485x570    276,450   -0.663    2.835   0.0000  ✓
GDP_SSP1_2100_normalized.tif             485x570    276,450   -0.663    2.849   0.0000  ✓
GDP_SSP2_2030_normalized.tif             485x570    276,450   -0.663    2.839   0.0000  ✓
GDP_SSP2_2050_normalized.tif             485x570    276,450   -0.663    2.827   0.0000  ✓
GDP_SSP2_2100_normalized.tif             485x570    276,450   -0.664    2.811   0.0000  ✓
GDP_SSP3_2030_normalized.tif             485x570    276,450   -0.663    2.834  -0.0000  ✓
GDP_SSP3_2050_normalized.tif             485x570    276,450   -0.663    2.831   0.0000  ✓
GDP_SSP3_2100_normalized.tif             485x570    276,450   -0.662    2.859  -0.0000 